In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

from inversion_utils import *
import utils

In [3]:
import torch
import numpy as np

In [4]:
SEED = 0
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

In [5]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# # MODEL_VERSION='2'
# MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# # MODEL_SIZE='9B'
# MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'



llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


Normal Test

In [7]:
# ["hemingway", "faulkner", "joyce", "austen", "woolf", "kafka", "stein", "cummings", "nabokov", "wilde", "twain", "wodehouse", "mccarthy", "carver", "bukowski", "márquez", "borges", "thompson", "lovecraft", "pratchett",]

In [7]:
author_map = {
        "hemingway": "Ernest Hemingway",
        "faulkner": "William Faulkner",
        "joyce": "James Joyce",
        "austen": "Jane Austen",
        "woolf": "Virginia Woolf",
        "kafka": "Franz Kafka",
        "stein": "Gertrude Stein",
        "cummings": "E.E. Cummings",
        "nabokov": "Vladimir Nabokov",
        "wilde": "Oscar Wilde",
        "twain": "Mark Twain",
        "wodehouse": "P.G. Wodehouse",
        "mccarthy": "Cormac McCarthy",
        "carver": "Raymond Carver",
        "bukowski": "Charles Bukowski",
        "márquez": "Gabriel García Márquez",
        "borges": "Jorge Luis Borges",
        "thompson": "Hunter S. Thompson",
        "lovecraft": "H.P. Lovecraft",
        "pratchett": "Terry Pratchett",
    }

In [8]:
prompts = ["How does a clock work?"] # "hemingway"

style = "hemingway"
style_source = "kafka"

In [ ]:
# sent = 'Write a response to the following statement in the literary style of {}. \nStatement: {}'.format(author_map[style], prompts[0])
sent = 'Write a response to the following statement in the literary style of {}. \nStatement: {}'.format(author_map[style_source], prompts[0])

messages = [
    {"role": "user", "content": sent}
]

# prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
tok_prompts = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).strip()

out = llm.tokenizer(tok_prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(llm.language_model.device)

outputs = llm.language_model.generate(**out, max_new_tokens=100, do_sample=False)
print(llm.tokenizer.decode(outputs[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Write a response to the following statement in the literary style of Franz Kafka. 
Statement: How does a clock work?assistant

The clock, a monolith of precision, a sentinel of time, its workings a mystery that has long fascinated and confounded me.  As I stand before its imposing face, I am struck by the futility of my inquiry.  How does a clock work?  The question itself seems a cruel jest, a taunt from the very fabric of existence.

I reach out a trembling hand, as if to grasp the elusive threads of its mechanism, but they slip through my fingers like sand.  The clock's gears, a labyrinthine maze of interlocking cogs and springs, seem to defy comprehension, a maddening puzzle that mocks my feeble attempts at understanding.

And yet, I am drawn to it, this enigmatic device, this keeper of the hours.  I am a moth to the flame, helpless to resist its siren call.  I peer into its depths, a window into the very 

In [ ]:
# Hemingway: "The clock. A machine of gears and springs, ticking away like a heartbeat. It's simple, really. A mainspring, coiled tight, stores energy. A series of gears, meshing together, release it. The balance wheel, a delicate dance, regulates the flow. It's a dance of metal and time, a symphony of gears and springs. The clock keeps ticking, a steady pulse, marking the passage of hours. It's a reminder that time waits for no man, that every tick is a step closer to the end."

# Kafka: The clock, a monolith of precision, a sentinel of time, its workings a mystery that has long fascinated and confounded me.  As I stand before its imposing face, I am struck by the futility of my inquiry.  How does a clock work?  The question itself seems a cruel jest, a taunt from the very fabric of existence. I reach out a trembling hand, as if to grasp the elusive threads of its mechanism, but they slip through my fingers like sand.  The clock's gears, a labyrinthine maze of interlocking cogs and springs, seem to defy comprehension, a maddening puzzle that mocks my feeble attempts at understanding. And yet, I am drawn to it, this enigmatic device, this keeper of the hours.  I am a moth to the flame, helpless to resist its siren call.  I peer into its depths, a window into the very heart of time itself, and what do I see?  A blur

In [9]:
# coef = 0.45
# coef = 0.55
# coef = 0.65
coef = 0.7 # llama
# coef = 0.75

# coef = 2.0
# coef = 2.5 # qwen

path = "../all_gitignore/xRFM/test/"

max_tokens = 200

partial = "The clock, a monolith of precision, a sentinel of time, its workings a mystery that has long fascinated and confounded me.  As I stand before its imposing face, I am struck by the futility of my inquiry.  How does a clock work?  The question itself seems a cruel jest, a taunt from the very fabric of existence. I reach out a trembling hand, as if to grasp the elusive threads of its mechanism, but they slip through my fingers like sand."

stylised_prompt = [f"Continue the following story, while maitaining the style:\n {partial}"]

s_controller = load_controller_translate(llm, style, style_source, path=path)

out = test_concept_vector(s_controller, concept=f"coef: {coef}, source: {style_source}, target: {style}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens, qwen=False, orig=True)
# out = test_concept_vector(s_controller, concept=f"coef: {coef}, source: {style_source}, target: {style}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Continue the following story, while maitaining the style:
 The clock, a monolith of precision, a sentinel of time, its workings a mystery that has long fascinated and confounded me.  As I stand before its imposing face, I am struck by the futility of my inquiry.  How does a clock work?  The question itself seems a cruel jest, a taunt from the very fabric of 